# word2vec を自分のコーパスで学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_word_vectors/word2vec_train.ipynb)

NLP クックブック 09-01 ／ 単語ベクトル

手元のコーパスから単語ベクトルを学習して、似た単語を調べたい。

In [1]:
!pip install -q gensim fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
text = "\n".join(aozora(base + f) for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"])  # 猫・坊っちゃん・こころ
sentences = [[w.surface for w in tagger(s)] for s in re.split(r"[。\n]", text) if s.strip()]
print(len(sentences), "文", sum(map(len, sentences)), "語")

16959 文 361852 語


In [3]:
from gensim.models import Word2Vec

model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, sg=1, negative=5, epochs=20, workers=2, seed=0)
print("語彙:", len(model.wv))
for w in ["先生", "猫", "東京", "学校"]:
    print(w, [(x, round(s, 2)) for x, s in model.wv.most_similar(w, topn=6)])

語彙: 4514
先生 [('私', 0.51), ('だい', 0.48), ('旧友', 0.47), ('奥', 0.47), ('未亡人', 0.46), ('名文', 0.44)]
猫 [('性質', 0.5), ('立場', 0.5), ('吾輩', 0.5), ('器量', 0.49), ('性分', 0.49), ('動物', 0.48)]
東京 [('仕方なし', 0.63), ('雑司ヶ谷', 0.62), ('寝床', 0.62), ('控', 0.61), ('一所', 0.59), ('宅', 0.59)]
学校 [('高等', 0.67), ('生徒', 0.64), ('物理', 0.63), ('大学', 0.58), ('泊っ', 0.57), ('規則', 0.55)]


In [4]:
model.save("w2v.model")                       # 追加学習できる形式
model.wv.save_word2vec_format("w2v.txt")      # 他のツールでも読めるテキスト形式
print(Word2Vec.load("w2v.model").wv["猫"][:5])

[-0.38791546 -0.60868746  0.24166787 -0.05187656  0.7624031 ]


- `sg=1` が skip-gram、`sg=0` が CBOW。小さいコーパスでは skip-gram + エポック多めがよい。
- 数十万語程度のコーパスでは品質は限定的。実用には数億語規模か、学習済みベクトルを使う。